# DPoP replay durability analysis

This notebook reads only the validated DPoP summary and raw event records. To analyze another run, update `SUMMARY` and `DATASETS` below. It does not modify raw data.

In [ ]:
from pathlib import Path
import csv, json
import matplotlib.pyplot as plt

SUMMARY = Path('results/dpop-durability/dpop-2026-11-19T07-46-29-011Z-55d0f618/summaries/results-table.csv')
DATASETS = [
    Path('results/dpop-durability/dpop-2026-11-19T07-46-29-011Z-55d0f618'),
    Path('results/dpop-durability/dpop-2026-11-19T07-49-40-957Z-d36a3dee'),
    Path('results/dpop-durability/dpop-repl-2026-11-19T08-16-16-028Z-a10c2e92'),
]
with SUMMARY.open() as f:
    summary = list(csv.DictReader(f))
summary

In [ ]:
for dataset in DATASETS:
    validation = json.loads((dataset / 'validation' / 'validation-summary.json').read_text())
    assert validation['valid_experiment_runs'] == 3
    with (dataset / 'raw' / 'requests.csv').open() as f:
        requests = list(csv.DictReader(f))
    print(dataset.name, len(requests), 'measured calls; validation:', validation['valid_experiment_runs'], 'valid runs')


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar([r['scenario'] for r in summary], [float(r['replay_acceptance_rate']) for r in summary])
ax.set_ylabel('Replay acceptance rate')
ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=40)
ax.set_title('Validated replay outcomes; sample size varies by condition')
fig.tight_layout()
plt.show()

The generated CSV/Markdown tables and PNG/PDF figures are produced by `scripts/analyze.py`. Inference is limited to this single-host study: standalone volatile Redis and two-node Redis replication with manual promotion. The replica is not a separate machine, and the results do not establish behavior under a true network partition, Sentinel, Redis Cluster, or MySQL.